# How do models find similarity and boundaries?
**Topics:** K-Nearest Neighbors (k-NN) · Support Vector Machines (SVM) · Kernel Trick


---
## 1. K-Nearest Neighbors (k-NN)

### What it is
- Non-parametric algorithm — no training phase, just stores data
- Classifies a new point by majority vote of its K nearest neighbors
- For regression: predicts the mean of K nearest neighbors' values

### Key intuition
- "You are who your neighbors are"
- Distance determines similarity — nearest neighbors = most similar training points
- No model is learned — prediction time does all the work

### Distance metrics

| Metric | Formula | Best for |
|---|---|---|
| Euclidean | sqrt(Σ(xᵢ - yᵢ)²) | Continuous features |
| Manhattan | Σ\|xᵢ - yᵢ\| | High dimensions, robust to outliers |
| Minkowski | (Σ\|xᵢ - yᵢ\|^p)^(1/p) | Generalization of above |
| Cosine | 1 - (x·y)/(‖x‖‖y‖) | Text, sparse data |

### When to use
- Small datasets where prediction speed is not critical
- Non-linear decision boundaries
- Simple baseline that is hard to beat on some tasks

### When not to use
- Large datasets — prediction is O(n) per query (slow)
- High-dimensional data — distances become meaningless (curse of dimensionality)
- Features on very different scales without normalization


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Always scale before k-NN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── k-NN Classifier ──
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', weights='uniform')
knn.fit(X_train_s, y_train)
print(f"k=5 test accuracy: {accuracy_score(y_test, knn.predict(X_test_s)):.3f}")

# Distance-weighted k-NN (closer neighbors have more influence)
knn_w = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn_w.fit(X_train_s, y_train)
print(f"k=5 weighted acc : {accuracy_score(y_test, knn_w.predict(X_test_s)):.3f}")

# Choosing K
print(f"\n{'K':<6} {'CV Accuracy'}")
for k in [1, 3, 5, 7, 11, 15, 21]:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(knn_k, X_train_s, y_train, cv=5, scoring='accuracy').mean()
    print(f"{k:<6} {score:.3f}")

# Get neighbors for a new point
distances, indices = knn.kneighbors(X_test_s[:1])
print(f"\nNearest neighbor indices  : {indices[0]}")
print(f"Distances to neighbors    : {distances[0].round(3)}")


### Interview Q&A

**How do you choose K?**
- Small K → complex boundary, high variance (overfitting)
- Large K → smooth boundary, high bias (underfitting)
- Use cross-validation to find optimal K — typically odd values to avoid ties

**Why must you scale features for k-NN?**
- Distance is computed across all features
- A feature with range [0, 1000] dominates one with range [0, 1]
- StandardScaler or MinMaxScaler required

**What is the curse of dimensionality in k-NN?**
- In high dimensions, all points become roughly equidistant
- Nearest neighbors are no longer meaningfully "near"
- k-NN degrades badly above ~20 features

### Gotchas
- k-NN has no training phase but prediction is O(n×d) — very slow on large datasets
- Use `KDTree` or `BallTree` (set algorithm='kd_tree') to speed up neighbor search
- weights='distance' generally outperforms uniform — closer neighbors should matter more


---
## 2. Support Vector Machines (SVM)

### What it is
- Finds the **maximum margin hyperplane** that best separates classes
- Support vectors = the data points closest to the decision boundary (they define it)
- Can handle non-linear boundaries using the kernel trick

### Key intuition
- Draw the widest possible "street" between two classes — the boundary is the center of the street
- Only the support vectors matter — removing other points doesn't change the boundary
- Larger margin → better generalization

### Hard margin vs soft margin
- **Hard margin** → no misclassification allowed — only works if data is linearly separable
- **Soft margin** → allows some misclassification (controlled by C parameter)
  - High C → narrow margin, few errors, risk of overfitting
  - Low C → wide margin, more errors allowed, more regularized

### When to use
- Small to medium datasets (< 100k samples)
- High-dimensional data (text classification)
- Clear margin of separation between classes

### When not to use
- Large datasets — training is O(n²) to O(n³), very slow
- Need probability outputs — SVM gives distances not probabilities by default
- Many overlapping classes — SVM struggles with noise near the boundary


In [ ]:
import numpy as np
from sklearn.svm import SVC, SVR
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# ── Linear SVM ──
X, y = make_classification(n_samples=500, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

svm_linear = SVC(kernel='linear', C=1.0)
svm_linear.fit(X_train_s, y_train)
print(f"Linear SVM accuracy    : {accuracy_score(y_test, svm_linear.predict(X_test_s)):.3f}")
print(f"Support vectors        : {svm_linear.n_support_}")

# ── RBF kernel SVM (non-linear) ──
X_nl, y_nl = make_moons(n_samples=400, noise=0.15, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_nl, y_nl, test_size=0.2)
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_tr, y_tr)
print(f"RBF SVM accuracy       : {accuracy_score(y_te, svm_rbf.predict(X_te)):.3f}")

# ── Tuning C and gamma ──
param_grid = {'C': [0.1, 1, 10], 'gamma': ['scale', 'auto', 0.1]}
grid = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5, scoring='accuracy')
grid.fit(X_tr, y_tr)
print(f"Best params            : {grid.best_params_}")
print(f"Best CV accuracy       : {grid.best_score_:.3f}")


### Interview Q&A

**What are support vectors?**
- The training points that lie on or within the margin
- They are the only points that define the decision boundary
- Removing non-support vectors does not change the model

**What does the C parameter control?**
- Regularization strength — tradeoff between margin width and misclassification
- High C → hard margin, fits training data closely, risk of overfitting
- Low C → wide margin, allows more errors, better generalization

**Why scale features for SVM?**
- SVM optimizes based on distances — unscaled features dominate
- Always StandardScaler before SVM

### Gotchas
- SVM is slow on large datasets — use LinearSVC for large linear problems (much faster)
- probability=True adds significant training cost (Platt scaling) — avoid unless needed
- GridSearch C and gamma together — they interact strongly


---
## 3. The Kernel Trick

### What it is
- A mathematical technique that allows SVM (and other algorithms) to operate in high-dimensional feature spaces without explicitly computing the transformation
- Replaces dot products with a kernel function K(x, z) = φ(x)·φ(z)
- Effectively maps data to a higher dimension where it becomes linearly separable

### Key intuition
- Data not linearly separable in 2D → transform to 3D where it is → find linear boundary there
- Kernel trick: compute the dot product in high-dim space without actually going there → computationally cheap

### Common kernels

| Kernel | Formula | When to use |
|---|---|---|
| Linear | x·z | Linearly separable, high-dim data (text) |
| RBF (Gaussian) | exp(-γ‖x-z‖²) | Most common default, non-linear boundaries |
| Polynomial | (x·z + c)^d | Polynomial relationships between features |
| Sigmoid | tanh(γx·z + c) | Rarely used |

### When to use
- Data is not linearly separable in original feature space
- RBF kernel is the default go-to for non-linear problems
- Linear kernel for high-dimensional sparse data (text/NLP)


In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Concentric circles — not linearly separable
X, y = make_circles(n_samples=400, noise=0.1, factor=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Compare kernels on non-linearly separable data
kernels = {
    'linear'     : SVC(kernel='linear', C=1.0),
    'rbf'        : SVC(kernel='rbf', C=1.0, gamma='scale'),
    'poly (d=3)' : SVC(kernel='poly', degree=3, C=1.0),
}

for name, model in kernels.items():
    model.fit(X_train_s, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_s))
    print(f"{name:<14}: {acc:.3f}")

# Manual RBF kernel intuition
def rbf_kernel(X, Z, gamma=1.0):
    # Computes pairwise RBF kernel matrix
    diff = X[:, None, :] - Z[None, :, :]       # (n, m, d)
    sq_dist = np.sum(diff**2, axis=2)           # (n, m)
    return np.exp(-gamma * sq_dist)

K = rbf_kernel(X_train_s[:5], X_train_s[:5], gamma=0.5)
print(f"\nRBF kernel matrix (5x5 sample):")
print(K.round(3))
print("Diagonal = 1.0 (each point is perfectly similar to itself)")


### Interview Q&A

**What problem does the kernel trick solve?**
- Explicitly mapping to high-dim space is computationally expensive (or infinite-dimensional for RBF)
- Kernel trick computes the inner product in that space directly using the original features → same result, much cheaper

**What does gamma control in the RBF kernel?**
- Controls the "reach" of each training point
- High gamma → each point influences only its immediate neighbors → complex boundary → overfitting
- Low gamma → each point influences a wider area → smoother boundary → underfitting

**Can the kernel trick be used outside SVM?**
- Yes — kernel PCA, kernel ridge regression, Gaussian Processes all use kernels
- Any algorithm that only needs inner products between samples can use kernel trick

### Gotchas
- RBF kernel with wrong gamma can perform like linear (gamma too low) or overfit completely (gamma too high)
- Kernel SVM is O(n²) to O(n³) — impractical for large datasets
- For large datasets use linear kernel with LinearSVC or switch to neural networks
